# QLoRA training on a Kaggle T4

Trains a LoRA adapter on Qwen3-1.7B (pinned to its Hub revision in `train/config.toml`) with
`train/train_qlora.py` (Unsloth backend, 4-bit base, fp16) and pushes it to the Hugging Face Hub.
Hyperparameters come only from `train/config.toml` at the pinned commit.

Before running:

- Settings: accelerator **GPU T4 x2** (the notebook uses one of the two), **Internet on**.
- Add-ons, Secrets: `HF_TOKEN` (a write token), `SCRUB_TERMS` (the scrub terms file, one term
  per line) and `REPO_URL` (the code repo's clone URL). The push is refused if any pushed file
  contains a term. No secret is printed, and the notebook stops before doing anything when one is
  missing or empty.
- `HUB_REPO` is set; the push creates the Hub repo private. Check the lines marked `# OWNER:` in
  the parameters cell (the training data).
- `REPO_COMMIT` is the full SHA of the commit to train from. Re-pin it to a commit that contains
  `DATASET`, or point `KAGGLE_DATASET` at an uploaded `train.jsonl` under `/kaggle/input`.
- `SMOKE_ONLY = True` runs `scripts/smoke_loop.sh --kaggle` (20 LoRA steps on Qwen3-0.6B, mock
  teacher and judge) instead of training and pushing.

Afterwards, download `/kaggle/working/run/train_log.json` and commit it next to the run's results
(R14); the adapter itself lives on the Hub.

In [ ]:
# The code repo is cloned at REPO_COMMIT; its clone URL is the REPO_URL Kaggle secret.
REPO_COMMIT = "3ff1c63efd9cb485a1de709bd2ffc4f56d4f3c42"
# OWNER: the train.jsonl to train on, relative to the repo at REPO_COMMIT.
DATASET = "data/datasets/v1/train.jsonl"
# OWNER: or a /kaggle/input/.../train.jsonl; when non-empty it replaces DATASET.
KAGGLE_DATASET = ""
SEED = 0
# The private Hub model repo the adapter is pushed to.
HUB_REPO = "joe-alfaro/groundcite-qwen3-1.7b"
OUT = "/kaggle/working/run"
SMOKE_ONLY = False

Secrets go into the environment of the training process (and the clone command) only; nothing is printed.

In [ ]:
import os
import tempfile

from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()


def required(label: str, read) -> str:
    """A Kaggle secret's value; stops with a clear message when it is not attached or empty."""
    try:
        value = read()
    except Exception:  # Kaggle raises a raw service error for a secret that is not attached
        value = ""
    if not value:
        raise RuntimeError(
            f"the Kaggle secret {label} is missing or empty: add it under Add-ons > Secrets "
            "and attach it to this notebook"
        ) from None
    return value


hf_token = required("HF_TOKEN", lambda: client.get_secret("HF_TOKEN"))
scrub_terms = required("SCRUB_TERMS", lambda: client.get_secret("SCRUB_TERMS"))
# The code repo's clone URL; it is kept out of every tracked file.
REPO_URL = required("REPO_URL", lambda: client.get_secret("REPO_URL"))
os.environ["HF_TOKEN"] = hf_token
# Outside /kaggle/working, so it is never saved as notebook output.
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as terms:
    terms.write(scrub_terms)
os.environ["SCRUB_TERMS_PATH"] = terms.name

Check the GPU, then clone the repo at the pinned commit and install around Kaggle's torch.

In [ ]:
import os

import torch

# Kaggle's "GPU T4 x2" shows two GPUs; the model runs on one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if not torch.cuda.is_available():
    raise RuntimeError("no CUDA GPU: set Settings > Accelerator to GPU T4 x2")
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
import re
import subprocess
import tempfile
from importlib import metadata
from urllib.parse import urlsplit

REPO_DIR = "/tmp/repo"


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


def clone(url: str, dest: str) -> None:
    """Clone without ever showing the URL: a private repo's URL can carry a token. git's error is
    kept for debugging, but git may quote the URL, its host or its path, so all are redacted."""
    result = subprocess.run(["git", "clone", "--quiet", url, dest], capture_output=True, text=True)
    if result.returncode != 0:
        parts = urlsplit(url)
        secret = {url, parts.netloc, parts.path, parts.path.strip("/"), *re.split(r"[@:/]+", url)}
        secret |= {parts.hostname or "", parts.username or "", parts.password or ""}
        detail = result.stderr or ""
        for piece in sorted((p for p in secret if len(p) > 2), key=len, reverse=True):
            detail = detail.replace(piece, "<redacted>")
        raise RuntimeError(
            f"git clone of the REPO_URL secret failed (exit {result.returncode}): {detail.strip()}"
        ) from None


clone(REPO_URL, REPO_DIR)
sh(f"git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}")
# An explicit pin list without torch, triton or nvidia-*: Kaggle's CUDA torch stays. A transitive
# requirement (xformers via Unsloth) could still pull in another torch, so the preinstalled torch
# stack is pinned as a constraint: pip then picks compatible versions or fails loudly.
pins = []
for name in ("torch", "torchvision", "torchaudio", "triton"):
    try:
        pins.append(f"{name}=={metadata.version(name)}\n")
    except metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as constraints:
    constraints.writelines(pins)
requirements = f"{REPO_DIR}/train/requirements-kaggle.txt"
sh(f"pip install --quiet -r {requirements} -c {constraints.name}")
sh(f"pip install --quiet --no-deps -e {REPO_DIR}")

Train, then push the adapter, model card and `train_log.json` in one Hub commit. With
`SMOKE_ONLY`, run the R5 smoke loop on the GPU instead.

In [ ]:
if SMOKE_ONLY:
    sh(f"cd {REPO_DIR} && bash scripts/smoke_loop.sh --kaggle {OUT}-smoke")
else:
    dataset = KAGGLE_DATASET or DATASET
    train = f"python train/train_qlora.py --dataset {dataset} --out {OUT} --seed {SEED}"
    sh(f"cd {REPO_DIR} && {train} --hub-repo {HUB_REPO}")

In [ ]:
import json

if not SMOKE_ONLY:
    with open(f"{OUT}/train_log.json", encoding="utf-8") as f:
        log = json.load(f)
    hub = log["hub"] or {}
    final_loss = log["loss_curve"][-1]["loss"] if log["loss_curve"] else float("nan")
    print(
        f"{hub.get('repo')}@{hub.get('commit_sha')} | code {log['git_sha']} | "
        f"config {log['config_sha256'][:12]} | data {log['dataset_sha256'][:12]} | "
        f"template {str(log['chat_template_sha256'])[:12]} | "
        f"{log['steps']} steps | final loss {final_loss:.4f}"
    )